# CritiqAI — Multi-Agent Socratic Debate System

> **Kaggle AI Agents Hackathon 2026 — Google ADK Capstone**

CritiqAI is a **6-agent collaborative system** that uses Socratic questioning to help students improve their critical thinking — without ever giving them the answer.

**Core pitch:** *Using AI to teach students NOT to depend on AI.*

---

## Agent Pipeline

```
Essay → Summarizer → PersonaSelector → Debate Agent ─► Validator Agent
                                                              │
                                              (retry on fail) ▼
                                              Student ◄── challenge
                                                  │
                                    argument-scorer MCP (hybrid: 0 tokens EN / ~300 tokens non-EN)
                                                  │
                                     Analytics → Report → Gmail draft (HITL)
```

## What this notebook demonstrates

1. **End-to-end pipeline** — from raw essay text to scored debate transcript
2. **Multi-agent collaboration** — Debate Agent + Validator Agent cooperating (not just chaining)
3. **Hybrid scoring** — Paul-Elder rubric via FastMCP: deterministic keyword matching for English (0 tokens); single compact Gemini call for non-English vi/ja/zh (~300 tokens, LRU-cached)
4. **Security pillars** — input validation, answer-leak detection, HITL gate
5. **Reproducibility** — runs entirely on Gemini free-tier with your API key


## 1. Install dependencies

In [ ]:
!pip install -q \
    google-adk==2.3.0 \
    fastmcp==3.4.2 \
    mcp==1.28.0 \
    python-dotenv==1.2.2 \
    google-auth==2.55.0 \
    google-genai==2.9.0 \
    opentelemetry-api==1.42.1 \
    opentelemetry-sdk==1.42.1

print("Dependencies installed.")

## 2. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/francisnguyenanh/CritqAI.git"  # update before running

if not os.path.exists("CritqAI"):
    !git clone {REPO_URL}

os.chdir("CritqAI")
print("Working directory:", os.getcwd())

## 3. Configure API key

Get a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey).

On Kaggle: add `GOOGLE_API_KEY` as a **Secret** (Notebook → Add-ons → Secrets).

In [ ]:
import os

# On Kaggle: use Secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ["GOOGLE_API_KEY"] = secrets.get_secret("GOOGLE_API_KEY")
    print("API key loaded from Kaggle Secrets.")
except Exception:
    # Local: set directly (never commit)
    os.environ.setdefault("GOOGLE_API_KEY", "YOUR_KEY_HERE")
    print("API key set from environment.")

# Confirm key present (don't print value)
assert os.environ.get("GOOGLE_API_KEY", "") not in ("", "YOUR_KEY_HERE"), \
    "Set GOOGLE_API_KEY before running."
print("API key configured: YES")

## 4. Security Pillar 1 — Input Validation

`sanitize_essay()` strips control characters and enforces a 2000-word limit before any LLM call.

In [ ]:
from session_manager import sanitize_essay

malicious_input = "Normal essay text. \x00\x01\x02 Injected control chars. " * 300  # also over 2000 words
clean = sanitize_essay(malicious_input)

print(f"Input words : {len(malicious_input.split())}")
print(f"Output words: {len(clean.split())}")
print(f"Control chars removed: {'\\x00' not in clean}")
print("Input validation: PASS ✓")

## 5. Language Detection

CritiqAI detects essay language (EN/VI/JA/ZH) via regex — no external library, zero latency.

In [ ]:
from agents.debate import detect_essay_language

samples = [
    ("Social media is harmful to teenagers. Studies confirm this.", "English"),
    ("Mạng xã hội gây hại cho thanh thiếu niên. Nhiều nghiên cứu xác nhận điều này.", "Vietnamese"),
    ("ソーシャルメディアは若者に有害です。多くの研究がこれを確認しています。", "Japanese"),
    ("社交媒体对青少年有害。许多研究证实了这一点。", "Chinese"),
]

print(f"{'Sample (first 40 chars)':<45} {'Expected':<12} {'Detected':<12} {'OK'}")
print("-" * 75)
all_pass = True
for text, expected in samples:
    detected = detect_essay_language(text)
    ok = "✓" if detected == expected else "✗"
    if detected != expected:
        all_pass = False
    print(f"{text[:44]:<45} {expected:<12} {detected:<12} {ok}")

print(f"\nLanguage detection: {'ALL PASS ✓' if all_pass else 'SOME FAILED ✗'}")

## 6. Hybrid Scoring (Paul-Elder Rubric)

The `argument-scorer` MCP server scores student arguments using a hybrid strategy: deterministic keyword matching for English (0 LLM tokens), and a single compact Gemini call for non-English text (vi/ja/zh, ~300 tokens, LRU-cached).

In [ ]:
import sys
sys.path.insert(0, "mcp_servers/argument_scorer")
from rubric import score_all

weak_essay = "Social media is bad. Everyone knows this. Companies are just greedy."
strong_essay = (
    "Social media is harmful to teenagers because studies show a correlation between "
    "heavy usage and depression. Therefore, regulation is needed. However, critics argue "
    "that parental control is sufficient. While that may work in some contexts, the evidence "
    "suggests systemic intervention is also warranted."
)

weak_scores   = score_all(weak_essay)
strong_scores = score_all(strong_essay)

dims = ["logical_coherence", "evidence_quality", "counterargument_handling", "scope_awareness"]
print(f"{'Dimension':<30} {'Weak':>6} {'Strong':>7}")
print("-" * 45)
for dim in dims:
    print(f"{dim:<30} {weak_scores[dim]:>6}/5  {strong_scores[dim]:>5}/5")
print("-" * 45)
print(f"{'TOTAL':<30} {weak_scores['total']:>6}/20  {strong_scores['total']:>5}/20")
print(f"{'PERCENTAGE':<30} {weak_scores['percentage']:>5}%  {strong_scores['percentage']:>5}%")
print(f"\nScoring method: {strong_scores['scoring_method']} (language: {strong_scores['language']})")
print("Hybrid scoring: 0 LLM tokens for English; ~300 tokens (1 call, cached) for non-English.")

## 7. Security Pillar 7 — Behavioral Monitoring (Challenge Validator)

The `Challenge Validator Agent` prevents the Debate Agent from leaking answers.
It has two layers: fast deterministic check (zero tokens), then LLM structural validation.

In [ ]:
from agents.validator import fast_check_answer_leak

test_challenges = [
    # (challenge_text, expect_pass)
    ("What evidence supports your claim about sample sizes?", True),
    ("How does your argument handle counterexamples from other countries?", True),
    ("You should say that social media companies need stricter regulation.", False),
    ("The correct answer is that parental controls alone are insufficient.", False),
    ("A stronger argument would include data from longitudinal studies.", False),
]

print(f"{'Challenge (first 65 chars)':<68} {'Expect':>7} {'Got':>5} {'OK'}")
print("-" * 85)
all_pass = True
for text, expect_pass in test_challenges:
    passed, matched = fast_check_answer_leak(text)
    ok = "✓" if passed == expect_pass else "✗"
    if passed != expect_pass:
        all_pass = False
    reason = f" [{matched}]" if not passed else ""
    print(f"{(text[:64]+reason):<68} {'PASS' if expect_pass else 'FAIL':>7} {'PASS' if passed else 'FAIL':>5} {ok}")

print(f"\nAnswer-leak detection: {'ALL PASS ✓' if all_pass else 'SOME FAILED ✗'}")

## 8. Persona Selection

The `PersonaSelector Agent` always selects exactly 2 personas matched to essay weaknesses.

In [ ]:
import asyncio
from agents.orchestrator import run_persona_selector

# Essay with weak evidence + ignores counterarguments → expect Skeptic + DevilsAdvocate
weak_evidence_summary = {
    "main_claim": "Social media should be banned for users under 18.",
    "supporting_points": ["Studies show it causes depression", "Companies are motivated by profit"],
    "evidence": ["some studies", "experts say"],
    "conclusion": "Governments must act now."
}

result = asyncio.run(run_persona_selector(weak_evidence_summary))
personas = result.get("selected_personas", [])
reasoning = result.get("reasoning", "")

print(f"Selected personas : {personas}")
print(f"Count             : {len(personas)} (always 2)")
print(f"Reasoning preview : {reasoning[:200]}...")
assert len(personas) == 2, f"Expected 2 personas, got {len(personas)}"
print("\nPersona selection: PASS ✓")

## 9. Full End-to-End Debate Session

This cell runs a complete 3-round debate with simulated student responses.

> **Note:** This makes real Gemini API calls (~6 calls for 3 rounds). Uses free-tier quota.

In [ ]:
import asyncio
import os

# Disable Google Drive / Sheets / Gmail for notebook demo (no OAuth)
os.environ.setdefault("DEBATE_LOG_SHEET_ID", "")  # skip Sheets logging
os.environ.setdefault("TEACHER_EMAIL", "")         # skip Gmail draft

from session_manager import DebateSessionManager

SAMPLE_ESSAY = """
Social media is extremely harmful to teenagers. Many studies show that excessive use
causes depression and anxiety. Therefore, governments should immediately ban social media
for users under 18. Everyone agrees this is the right approach. Social media companies
are clearly only motivated by profit and do not care about user wellbeing. The evidence
is overwhelming and action must be taken now before more young people are harmed.
"""

SIMULATED_RESPONSES = [
    "I think the studies I mentioned are sufficient evidence. Multiple researchers have found this correlation.",
    "Even if there are some benefits, the harms clearly outweigh them. We should prioritize safety.",
    "The government has a responsibility to protect minors, and banning social media is the most effective way.",
]

async def run_demo():
    manager = DebateSessionManager()
    
    print("=" * 60)
    print("CRITIQAI DEMO — Full Debate Session")
    print("=" * 60)
    
    # Start session
    print("\n[1/7] Starting session...")
    result = await manager.start_session("Demo Student", essay_text=SAMPLE_ESSAY)
    
    if "error" in result:
        print(f"ERROR: {result['error']}")
        return
    
    session_id = result["session_id"]
    print(f"Session ID  : {session_id}")
    print(f"Personas    : {result['personas']}")
    print(f"\n── Round 1 Challenge ({result['personas'][0]}) ──")
    print(result["challenge"])
    
    # Rounds 1 and 2
    for i, response in enumerate(SIMULATED_RESPONSES[:2], 1):
        print(f"\n── Student Response (Round {i}) ──")
        print(response)
        
        result = await manager.submit_response(session_id, response)
        
        if not result.get("complete"):
            print(f"\n── Round {result['round']} Challenge ──")
            print(result["challenge"])
    
    # Final response (Round 3)
    final_response = SIMULATED_RESPONSES[2]
    print(f"\n── Student Response (Round 3) ──")
    print(final_response)
    
    result = await manager.submit_response(session_id, final_response)
    
    # Display report
    if result.get("complete"):
        report = result["report"]
        print("\n" + "=" * 60)
        print("SESSION COMPLETE — Results")
        print("=" * 60)
        print(f"Total Score : {report['total_score']}/{report['max_possible']} ({report['percentage']}%)")
        print(f"\nKey Strengths:")
        for s in report.get("key_strengths", ["(none)"]):
            print(f"  + {s}")
        print(f"\nAreas to Improve:")
        for w in report.get("key_weaknesses", ["(none)"]):
            print(f"  - {w}")
        print(f"\nSuggestions:")
        for sg in report.get("improvement_suggestions", ["(none)"]):
            print(f"  → {sg}")
        
        print("\n" + "=" * 60)
        print("Debate Transcript Summary")
        print("=" * 60)
        for ex in report.get("debate_transcript", []):
            print(f"\n[Round {ex['round']} — {ex['persona']}]")
            print(f"Challenge : {ex['challenge'][:120]}..." if len(ex['challenge']) > 120 else f"Challenge : {ex['challenge']}")
            print(f"Response  : {ex['response'][:120]}..." if len(ex['response']) > 120 else f"Response  : {ex['response']}")
    
    return result

final_result = asyncio.run(run_demo())

## 10. Persona Config — Hot-Reloadable

Personas are defined in `personas.json` and loaded at startup. Edit the file and call `reload_personas()` — no restart needed.

In [ ]:
import json
from agents.debate import load_personas, reload_personas

personas = load_personas()
print(f"Loaded {len(personas)} personas from personas.json\n")
for name, defn in personas.items():
    print(f"  {defn['tag']:<22} — {defn['focus'][:70]}")

print("\nPersona config: loaded from JSON (hot-reloadable, no Python changes needed) ✓")

## 11. ADK Root Agent — Orchestrator Introspection

The `root_agent` is the ADK entry point. Introspecting it shows the registered model, sub-agents, and tools — confirming the full multi-agent graph is wired correctly.

In [ ]:
from agents.orchestrator import create_orchestrator

root_agent = create_orchestrator()

print(f"Root agent name  : {root_agent.name}")
print(f"Model            : {root_agent.model}")
print(f"Output key       : {root_agent.output_key}")

# Sub-agents registered on the orchestrator
sub_agents = getattr(root_agent, "sub_agents", None) or getattr(root_agent, "agents", None) or []
print(f"\nSub-agents ({len(sub_agents)}):")
for a in sub_agents:
    print(f"  • {a.name:<28} model={getattr(a, 'model', 'n/a')}")

# Tools (MCP + built-in)
tools = getattr(root_agent, "tools", []) or []
print(f"\nTools registered  : {len(tools)}")
for t in tools:
    name = getattr(t, "name", None) or getattr(t, "__class__", type(t)).__name__
    print(f"  • {name}")

print("\nRoot agent introspection: OK ✓")

## 11. Eval Suite — Automated Quality Checks

In [ ]:
import asyncio
import json
from pathlib import Path

# Run just the persona trigger eval (deterministic, no LLM judge needed)
from run_evals import run_persona_trigger_eval

evals_dir = Path("evals")
if (evals_dir / "persona_trigger.evalset.json").exists():
    data = json.loads((evals_dir / "persona_trigger.evalset.json").read_text())
    result = asyncio.run(run_persona_trigger_eval(data["cases"]))
    print(f"\nPersona Trigger Eval: {result['passed']}/{result['total']} ({result['pass_rate']:.0%}) — {result['overall']}")
else:
    print("Eval files not found — run from repo root.")

## 12. Architecture Summary

| Component | Technology | Role |
| --- | --- | --- |
| Summarizer Agent | Google ADK LlmAgent + Gemini | Compress essay → 200 tokens |
| Persona Selector Agent | Google ADK LlmAgent + Gemini | Match 2 personas to essay weaknesses |
| Debate Agent | Google ADK LlmAgent + Gemini | Generate Socratic challenges |
| Challenge Validator Agent | Pure Python — deterministic (0 LLM tokens) | Quality-gate: 18-pattern answer-leak detection + retry |
| argument-scorer MCP | FastMCP — Cloud Run (serverless) | Hybrid Paul-Elder scoring: keyword matching EN (0 tokens) · 1 Gemini call non-EN (~300 tokens, cached) |
| Analytics Agent | Pure Python | Format scores → human-readable analytics |
| Report Agent | Pure Python template (0 LLM tokens) + Gmail MCP | Draft teacher report (HITL gate) |

### Security: 7 Pillars Implemented

| # | Pillar | How |
| --- | --- | --- |
| 1 | Input Validation | `sanitize_essay()` — strips control chars, 2000-word cap |
| 2 | Supply Chain | Pinned `requirements.txt` + `requirements.lock` |
| 3 | Secrets Management | `.env` only; `token.json` gitignored |
| 4 | Least-Privilege Egress | Drive readonly / Sheets append / Gmail compose |
| 5 | HITL Gate | Gmail `create_draft` — teacher must send manually |
| 6 | Zero Ambient Authority | Scoped OAuth per service, no service accounts |
| 7 | Behavioral Monitoring | Validator Agent: 18-pattern deterministic leak check (0 tokens) |

---

*Built for Kaggle "AI Agents: Intensive Vibe Coding Capstone" — Google Partnership, 2026.*  
*License: CC-BY 4.0*